# Marketing LLM — Adaptive LoRA Fine-Tune on Kaggle

**Auto-detects GPU and picks the optimal path:**
- **T4 ×2 / V100 / A100** (compute capability ≥ 7.0) → Unsloth + Llama 3.1 **8B** + 4-bit QLoRA
- **P100** (compute capability 6.0) → vanilla transformers + Llama 3.2 **3B** + fp16 LoRA

Either path saves a LoRA adapter and pushes to Hugging Face Hub. Output repo name reflects the actual model trained.

Set HF_TOKEN as a Kaggle Secret (Add-ons → Secrets) to enable HF push.

In [ ]:
# ── GPU detection ───────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), '❌ No GPU detected'

props = torch.cuda.get_device_properties(0)
GPU_NAME = torch.cuda.get_device_name(0)
CC_MAJOR = props.major
CC_MINOR = props.minor
VRAM_GB = props.total_memory / 1e9

USE_UNSLOTH = CC_MAJOR >= 7
PATH = 'unsloth-8b' if USE_UNSLOTH else 'vanilla-3b'

print(f'GPU: {GPU_NAME} — {VRAM_GB:.1f} GB — cc {CC_MAJOR}.{CC_MINOR}')
print(f'Path: {PATH}')
print(f'  → Llama 3.1 8B + Unsloth + 4-bit QLoRA' if USE_UNSLOTH else '  → Llama 3.2 3B + vanilla transformers + fp16 LoRA')
print(f'Torch CUDA: {torch.version.cuda}, torch {torch.__version__}')

In [ ]:
# ── Install dependencies based on path ──────────────────────────
import subprocess, sys
def pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])
def pip_uninstall(*a):
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', *a], check=False)

pip('--upgrade', 'pip')

if USE_UNSLOTH:
    # T4×2 / V100 / A100 path — Unsloth + 4-bit
    pip('--upgrade', 'torchao>=0.16.0')
    pip_uninstall('unsloth', 'unsloth_zoo', 'bitsandbytes')
    pip('--upgrade', '--no-cache-dir', 'bitsandbytes>=0.46.1')
    pip('--upgrade', 'transformers>=4.49.0,<4.55.0')
    pip('--upgrade', 'peft>=0.14.0,<0.16.0', 'trl>=0.12.0,<0.13.0', 'accelerate>=1.2.0')
    pip('--upgrade', '--no-cache-dir',
        'git+https://github.com/unslothai/unsloth.git@main',
        'git+https://github.com/unslothai/unsloth_zoo.git@main')
else:
    # P100 path — vanilla transformers, AVOID torchao entirely (Kaggle ships 0.10
    # which doesn't have Pascal kernels; newer 0.16+ has CUDA kernels but they
    # don't include cc 6.0 binaries either). Use an older peft that doesn't try
    # to import torchao at all.
    pip_uninstall('torchao')
    pip('--upgrade', 'transformers>=4.46.0,<4.50.0')
    pip('--upgrade', 'peft>=0.11.0,<0.14.0')
    pip('--upgrade', 'trl>=0.10.0,<0.13.0', 'accelerate>=1.0.0')

pip('datasets', 'huggingface_hub')
print(f'✓ Install complete for {PATH}')

In [ ]:
# ── Clone training data from repo ───────────────────────────────
import os, json
if not os.path.exists('/kaggle/working/Marketing'):
    os.system('git clone -q https://github.com/amittomar-hue/Marketing.git /kaggle/working/Marketing')

DATA_PATH = '/kaggle/working/Marketing/training/data/marketing_sft.jsonl'
with open(DATA_PATH) as f:
    examples = [json.loads(l) for l in f]
print(f'Loaded {len(examples)} training examples')

In [ ]:
# ── Load base model (path-dependent) ────────────────────────────
import torch

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
    MODEL_SHORT = '8b'
    max_seq_length = 2048
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    # Try unsloth's mirror first (always public), fall back to Meta's gated repo
    candidates = ['unsloth/Llama-3.2-3B-Instruct', 'meta-llama/Llama-3.2-3B-Instruct']
    model, tokenizer = None, None
    for name in candidates:
        try:
            print(f'Trying {name}...')
            tokenizer = AutoTokenizer.from_pretrained(name)
            model = AutoModelForCausalLM.from_pretrained(
                name,
                torch_dtype=torch.float16,
                device_map='auto',
                low_cpu_mem_usage=True,
            )
            MODEL_NAME = name
            MODEL_SHORT = '3b'
            max_seq_length = 1024
            print(f'✓ Loaded {name}')
            break
        except Exception as e:
            print(f'  failed: {str(e)[:200]}')
    assert model is not None, 'Could not load 3B model from any candidate'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# ── Attach LoRA adapters ────────────────────────────────────────
if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    model = FastLanguageModel.get_peft_model(
        model, r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=16, lora_dropout=0.05, bias='none',
        use_gradient_checkpointing='unsloth', random_state=42,
    )
else:
    from peft import LoraConfig, get_peft_model, TaskType
    cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        bias='none',
    )
    model = get_peft_model(model, cfg)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# ── Format dataset with Llama 3 chat template ───────────────────
from datasets import Dataset

SYSTEM_PROMPT = 'You are Marketing LLM, an enterprise-grade marketing assistant. Be direct, specific, and data-driven. Format responses in clean markdown.'

def format_example(ex):
    messages = [
        {'role':'system','content':SYSTEM_PROMPT},
        {'role':'user','content':ex['instruction']},
        {'role':'assistant','content':ex['output']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': text}

raw = Dataset.from_list(examples)
ds = raw.map(format_example, remove_columns=raw.column_names)
ds = ds.train_test_split(test_size=0.05, seed=42)
print(f'Train: {len(ds["train"])}, Eval: {len(ds["test"])}')

In [ ]:
# ── Train (path-dependent hyperparams) ──────────────────────────
from transformers import TrainingArguments

common_args = dict(
    output_dir='/kaggle/working/checkpoints',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_steps=5,
    logging_steps=5,
    eval_strategy='steps',
    eval_steps=20,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=42,
    report_to='none',
)

if USE_UNSLOTH:
    from trl import SFTTrainer
    from unsloth import is_bfloat16_supported
    args = TrainingArguments(
        **common_args,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim='adamw_8bit',
    )
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=ds['train'], eval_dataset=ds['test'],
        dataset_text_field='text', max_seq_length=max_seq_length,
        dataset_num_proc=2, packing=False, args=args,
    )
else:
    from transformers import Trainer, DataCollatorForLanguageModeling
    # Tokenize for vanilla Trainer
    def tok(ex):
        out = tokenizer(ex['text'], truncation=True, max_length=max_seq_length, padding=False)
        out['labels'] = out['input_ids'].copy()
        return out
    tokenized = ds.map(tok, remove_columns=['text'])
    args = TrainingArguments(
        **common_args,
        fp16=True, bf16=False,           # P100 = fp16 only
        optim='adamw_torch',
        gradient_checkpointing=True,
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=tokenized['train'], eval_dataset=tokenized['test'],
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )

stats = trainer.train()
print(f'\n✓ Training complete. Final loss: {stats.training_loss:.4f}')

In [ ]:
# ── Save adapter ────────────────────────────────────────────────
import gc, torch
gc.collect(); torch.cuda.empty_cache()

ADAPTER_DIR = f'/kaggle/working/marketing-llm-{MODEL_SHORT}-lora'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
import os
os.system(f'du -sh {ADAPTER_DIR}')
print(f'✓ Saved to {ADAPTER_DIR}')

In [ ]:
# ── Push to Hugging Face Hub ────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    hf_token = None
    print(f'HF_TOKEN secret not available: {e}')

if hf_token:
    from huggingface_hub import login, HfApi
    login(token=hf_token)
    api = HfApi(token=hf_token)
    REPO_NAME = f'dmoop/marketing-llm-{MODEL_SHORT}-lora'
    try:
        api.create_repo(repo_id=REPO_NAME, exist_ok=True, private=False)
        model.push_to_hub(REPO_NAME, token=hf_token)
        tokenizer.push_to_hub(REPO_NAME, token=hf_token)
        print(f'\n✓ Pushed to https://huggingface.co/{REPO_NAME}')
    except Exception as e:
        print(f'HF push failed: {e}')
        print(f'Adapter saved at {ADAPTER_DIR} for manual download via Kaggle output API')
else:
    print(f'Adapter saved at {ADAPTER_DIR} — no HF push attempted')